In [ ]:
import os
train_data_path = '../data/train/NIR/'
sets = os.listdir(train_data_path)

preview_count = 1
sets = sets[:preview_count]

In [ ]:
import numpy as np

def add_salt_and_pepper_noise(image, salt_prob=0.01, pepper_prob=0.01):
    """
    Add salt and pepper noise to a 16-bit depth image.
    
    Args:
    image (numpy.ndarray): Input image (384x384, 16-bit depth).
    salt_prob (float): Probability of adding salt noise (default: 0.02).
    pepper_prob (float): Probability of adding pepper noise (default: 0.02).
    
    Returns:
    numpy.ndarray: Image with salt and pepper noise.
    """
    
    # Create a copy of the input image
    noisy_image = np.copy(image)
    
    # Get the maximum value for 16-bit depth
    max_value = np.iinfo(image.dtype).max  # This will be 65535 for 16-bit
    
    # Generate salt noise
    salt_mask = np.random.random(image.shape[:2]) < salt_prob
    noisy_image[salt_mask] = max_value
    
    # Generate pepper noise
    pepper_mask = np.random.random(image.shape[:2]) < pepper_prob
    noisy_image[pepper_mask] = 0
    
    return noisy_image

In [ ]:
import numpy as np
from scipy.ndimage import gaussian_filter

def apply_gaussian_blur(image, sigma=1.0):
    """
    Apply Gaussian blur to a 16-bit depth image.
    
    Args:
    image (numpy.ndarray): Input image (384x384, 16-bit depth).
    sigma (float): Standard deviation for Gaussian kernel. Default is 1.0.
    
    Returns:
    numpy.ndarray: Blurred image with the same dtype as input.
    """
    # Apply Gaussian filter
    blurred = gaussian_filter(image, sigma=sigma)
    
    # Ensure the output is of the same dtype as input
    return blurred.astype(image.dtype)

In [ ]:
import numpy as np

def add_row_column_noise(image, row_prob=0.02, col_prob=0.02):
    """
    Add noise by setting entire rows or columns to 0 or max value.
    
    Args:
    image (numpy.ndarray): Input image (384x384, 16-bit depth).
    row_prob (float): Probability of affecting a row (default: 0.01).
    col_prob (float): Probability of affecting a column (default: 0.01).
    
    Returns:
    numpy.ndarray: Image with row and column noise.
    """
    # Create a copy of the input image
    noisy_image = image.copy()
    
    # Get the maximum value for the image's data type
    max_value = np.iinfo(image.dtype).max
    
    height, width = image.shape[:2]
    
    # Generate row noise
    for i in range(height):
        if np.random.random() < row_prob:
            noisy_image[i, :] = 0 if np.random.random() < 0.5 else max_value
    
    # Generate column noise
    for j in range(width):
        if np.random.random() < col_prob:
            noisy_image[:, j] = 0 if np.random.random() < 0.5 else max_value
    
    return noisy_image

In [ ]:
import numpy as np

def apply_angled_sinusoidal_brightening(image, amplitude=0.2, frequency=1):
    """
    Apply sinusoidal brightening to an image at a random angle with a random start phase.
   
    Args:
    image (numpy.ndarray): Input image (384x384, 16-bit depth).
    amplitude (float): Amplitude of the sine wave, controls intensity of brightening (0 to 1).
    frequency (float): Frequency of the sine wave, controls how often brightening occurs.
   
    Returns:
    numpy.ndarray: Image with angled sinusoidal brightening applied.
    """
    # Create a copy of the input image
    brightened_image = image.copy().astype(np.float64)
   
    height, width = image.shape[:2]
    max_value = np.iinfo(image.dtype).max
   
    # Generate a random angle in radians
    angle = np.random.uniform(0, 2 * np.pi)
    
    # Generate a random phase shift in radians
    phase = np.random.uniform(0, 2 * np.pi)
   
    # Create coordinate matrices
    y, x = np.ogrid[:height, :width]
   
    # Calculate the distance along the angled direction
    distance = x * np.cos(angle) + y * np.sin(angle)
   
    # Generate the sinusoidal pattern
    max_distance = np.sqrt(width**2 + height**2)
    sinusoidal_pattern = amplitude * np.sin(2 * np.pi * frequency * distance / max_distance + phase) + 1
   
    # Apply the pattern to the image
    brightened_image *= sinusoidal_pattern
   
    # Clip values to ensure they're within the valid range
    brightened_image = np.clip(brightened_image, 0, max_value)
   
    return brightened_image.astype(image.dtype)

In [ ]:
def print_image(titled_images, title, max, zoom=0.4):
    """
    Print a list of images.
    
    Args:
    images (list): List of images.
    title (str): Title of the plot.
    zoom (float): Zoom factor (default: 0.2). How much of the image will be visible (top-left corner).
    """
    
    import matplotlib.pyplot as plt
    
    fig, axs = plt.subplots(1, len(titled_images), figsize=(28, 5), dpi=300)
    fig.suptitle(title)

    for i, titled_image in enumerate(titled_images):
        (title, image) = titled_image
        width_visible = int(image.shape[1] * zoom)
        height_visible = int(image.shape[0] * zoom)
        axs[i].imshow(image[:height_visible, :width_visible], vmax=max)
        axs[i].set_title(title)

    plt.show()

In [ ]:
import imageio
import matplotlib.pyplot as plt

for set in sets:
    set_path = os.path.join(train_data_path, set)
    lr_filename = 'LR000.png'
    hr_filename = 'HR.png'

    lr_path = os.path.join(set_path, lr_filename)
    hr_path = os.path.join(set_path, hr_filename)

    lr = imageio.imread(lr_path)
    hr = imageio.imread(hr_path)

    # Get the max HR value for normalizing the display
    max_hr_value = np.iinfo(hr.dtype).max

    salt_and_pepper_noise = add_salt_and_pepper_noise(hr)

    gaussian_blur = apply_gaussian_blur(hr, sigma=0.5)

    row_column_noise = add_row_column_noise(hr)

    sinusoidal_brightening = apply_angled_sinusoidal_brightening(hr, amplitude=0.3, frequency=8)

    titled_images = [
        ('LR', lr),
        ('HR', hr),
        ('Salt and Pepper Noise', salt_and_pepper_noise),
        ('Gaussian Blur', gaussian_blur),
        ('Row and Column Noise', row_column_noise),
        ('Sinusoidal Brightening', sinusoidal_brightening)
    ]

    print_image(titled_images, set, max_hr_value)
